#### Error Mitigated QSVM (ZNE + REM) - Lung Cancer - Consistent with Ideal/Noisy

This notebook implements two error mitigation techniques:
1. **Zero-Noise Extrapolation (ZNE)**: Extrapolates results from multiple noise scales to estimate zero-noise expectation
2. **Readout Error Mitigation (REM)**: Corrects measurement errors using calibration-based mitigation matrix

In [1]:
!pip install qiskit qiskit-machine-learning qiskit-aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.1/263.1 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 60.1 MB/s eta 0:00:00


In [2]:
import qiskit, qiskit_aer, qiskit_machine_learning
print("Qiskit:", qiskit.__version__)
print("Aer:", qiskit_aer.__version__)
print("QML:", qiskit_machine_learning.__version__)

Qiskit: 2.2.3
Aer: 0.17.2
QML: 0.9.0


In [3]:
# To ensure reproducibility of results
from qiskit_machine_learning.utils import algorithm_globals
algorithm_globals.random_seed = 12345

In [4]:
# --- Import Libraries ---
import pandas as pd
import numpy as np
import time
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, recall_score, balanced_accuracy_score
from scipy.stats import chi2_contingency

In [5]:
# --- Qiskit Imports ---
from qiskit.circuit.library import ZZFeatureMap
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel

##### Load and Preprocess Dataset

In [6]:
# --- Load Lung Cancer Dataset ---
lung_cancer_column_names = ['label'] + [f'attr_{i}' for i in range(1, 57)]
# file_path = r'C:\Users\User\Documents\MyProjects\FYP_ResearchProject\data\lung+cancer\lung-cancer.data'
file_path_lung = '/kaggle/input/lungcancer-qsvm/lung-cancer.data'

# Read data, treating "?" as missing values
df = pd.read_csv(file_path_lung, header=None, names=lung_cancer_column_names, na_values=['?'])

print(f"Original shape of Lung Cancer data: {df.shape}")

# Mode imputation for missing values
modes = df.mode().iloc[0]
df.fillna(modes, inplace=True)
print(f"Total missing values after imputation: {df.isnull().sum().sum()}")

# Binary label: Class 1 -> 0, Others -> 1
df['label_binary'] = df['label'].apply(lambda x: 0 if x == 1 else 1)

print(f"\nClass distribution:")
print(df['label_binary'].value_counts())

Original shape of Lung Cancer data: (32, 57)
Total missing values after imputation: 0

Class distribution:
label_binary
1    23
0     9
Name: count, dtype: int64


In [7]:
# --- Feature Selection Helper: Cramér's V ---
def cramers_v(x, y):
    """Calculate Cramér's V statistic for categorical-categorical association."""
    confusion_matrix = pd.crosstab(x, y)
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
    rcorr = r - ((r-1)**2)/(n-1)
    kcorr = k - ((k-1)**2)/(n-1)
    if min((kcorr-1), (rcorr-1)) == 0:
        return 0
    return np.sqrt(phi2corr / min((kcorr-1), (rcorr-1)))

print("Cramér's V function ready for feature selection.")

Cramér's V function ready for feature selection.


##### Noise Model Factory Functions (ZNE + REM)

In [8]:
# Base error rates (realistic NISQ device)
NOISE_CONFIGS = {
    'low': {'p_1q': 0.0001, 'p_2q': 0.001, 'p_ro': 0.002},
    'standard': {'p_1q': 0.001,  'p_2q': 0.01,  'p_ro': 0.02},
    'high': {'p_1q': 0.005,  'p_2q': 0.05,  'p_ro': 0.10}
}

def get_scaled_noise_model(scale_factor=1.0, level='standard', include_readout=True):
    """
    Build a noise model with scaled error probabilities for ZNE.
    Uses formula: p_scaled = 1 - (1-p)^scale_factor
    """
    config = NOISE_CONFIGS.get(level, NOISE_CONFIGS['standard'])
    
    p_1q = config['p_1q']
    p_2q = config['p_2q']
    p_ro = config['p_ro']

    p_1q_scaled = 1 - (1 - p_1q)**scale_factor
    p_2q_scaled = 1 - (1 - p_2q)**scale_factor
    p_ro_scaled = 1 - (1 - p_ro)**scale_factor
    
    noise_model = NoiseModel()
    noise_model.add_all_qubit_quantum_error(depolarizing_error(p_1q_scaled, 1), ['u1', 'u2', 'u3'])
    noise_model.add_all_qubit_quantum_error(depolarizing_error(p_2q_scaled, 2), ['cx'])
    
    if include_readout:
        readout_error = ReadoutError([[1 - p_ro_scaled, p_ro_scaled], [p_ro_scaled, 1 - p_ro_scaled]])
        noise_model.add_all_qubit_readout_error(readout_error)
    
    backend = AerSimulator(noise_model=noise_model, seed_simulator=12345)
    pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
    
    return noise_model, backend, pm, config

print("Noise model factory function ready!")
print("Available Levels: 'low', 'standard', 'high'")

Noise model factory function ready!
Available Levels: 'low', 'standard', 'high'


In [9]:
# ==========================================
# READOUT ERROR MITIGATION (REM) FUNCTIONS
# ==========================================

def build_rem_matrix(p_ro, n_qubits):
    """
    Build the readout error mitigation (inverse calibration) matrix.
    """
    A_1q = np.array([[1 - p_ro, p_ro],
                     [p_ro, 1 - p_ro]])
    
    A_full = A_1q
    for _ in range(n_qubits - 1):
        A_full = np.kron(A_full, A_1q)
    
    try:
        rem_matrix = np.linalg.inv(A_full)
    except np.linalg.LinAlgError:
        rem_matrix = np.linalg.pinv(A_full)
    
    return rem_matrix


def apply_rem_to_kernel(kernel_matrix, p_ro, n_qubits):
    """
    Apply Readout Error Mitigation to a kernel matrix.
    """
    correction_factor = (1 - 2 * p_ro) ** n_qubits
    
    if abs(correction_factor) < 1e-10:
        print("Warning: Correction factor too small, using raw kernel")
        return kernel_matrix
    
    bias = 0.5 * (1 - correction_factor)
    corrected_kernel = (kernel_matrix - bias) / correction_factor
    
    corrected_kernel = np.clip(corrected_kernel, 0, 1)
    
    if kernel_matrix.shape[0] == kernel_matrix.shape[1]:
        np.fill_diagonal(corrected_kernel, 1.0)
    
    return corrected_kernel


print("REM (Readout Error Mitigation) functions ready!")

REM (Readout Error Mitigation) functions ready!


##### Experiment Configurations (ZNE + REM)

In [10]:
# ==========================================
# EXPERIMENT CONFIGURATIONS (ZNE + REM COMBINED)
# LUNG CANCER DATASET (OPTIMIZED)
# ==========================================
#
# Strategy: Focus on ZNE+REM combined (best mitigation)
# Use 8192 shots as baseline (good for small dataset)
# Include ablation studies (ZNE-only, REM-only, None)
#
# ==========================================

experiments = [
    # =====================================================
    # SECTION A: ZNE+REM COMBINED (PRIMARY EXPERIMENTS)
    # =====================================================
    
    # --- EXP 1: Dimensionality Effect (ZNE+REM) ---
    {'id': 'ZNEREM_4feat',   'k_features': 4,  'shots': 8192, 'reps': 2, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'ZNE+REM'},
    {'id': 'ZNEREM_6feat',   'k_features': 6,  'shots': 8192, 'reps': 2, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'ZNE+REM'},
    {'id': 'ZNEREM_8feat',   'k_features': 8,  'shots': 8192, 'reps': 2, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'ZNE+REM'},
    {'id': 'ZNEREM_10feat',  'k_features': 10, 'shots': 8192, 'reps': 2, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'ZNE+REM'},

    # --- EXP 2: Shot Noise Effect (ZNE+REM) ---
    {'id': 'ZNEREM_4096shots',  'k_features': 8, 'shots': 4096,  'reps': 2, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'ZNE+REM'},
    {'id': 'ZNEREM_8192shots',  'k_features': 8, 'shots': 8192,  'reps': 2, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'ZNE+REM'},
    {'id': 'ZNEREM_16384shots', 'k_features': 8, 'shots': 16384, 'reps': 2, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'ZNE+REM'},

    # --- EXP 3: Reps Effect (ZNE+REM) ---
    {'id': 'ZNEREM_Reps1', 'k_features': 8, 'shots': 8192, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'ZNE+REM'},
    {'id': 'ZNEREM_Reps2', 'k_features': 8, 'shots': 8192, 'reps': 2, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'ZNE+REM'},
    {'id': 'ZNEREM_Reps3', 'k_features': 8, 'shots': 8192, 'reps': 3, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'ZNE+REM'},

    # --- EXP 4: Entanglement Effect (ZNE+REM) ---
    {'id': 'ZNEREM_Linear',   'k_features': 8, 'shots': 8192, 'reps': 2, 'entanglement': 'linear',   'noise_level': 'standard', 'mitigation': 'ZNE+REM'},
    {'id': 'ZNEREM_Circular', 'k_features': 8, 'shots': 8192, 'reps': 2, 'entanglement': 'circular', 'noise_level': 'standard', 'mitigation': 'ZNE+REM'},
    {'id': 'ZNEREM_Full',     'k_features': 8, 'shots': 8192, 'reps': 2, 'entanglement': 'full',     'noise_level': 'standard', 'mitigation': 'ZNE+REM'},

    # --- EXP 5: Noise Level Effect (ZNE+REM) ---
    {'id': 'ZNEREM_LowNoise',  'k_features': 8, 'shots': 8192, 'reps': 2, 'entanglement': 'linear', 'noise_level': 'low',      'mitigation': 'ZNE+REM'},
    {'id': 'ZNEREM_StdNoise',  'k_features': 8, 'shots': 8192, 'reps': 2, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'ZNE+REM'},
    {'id': 'ZNEREM_HighNoise', 'k_features': 8, 'shots': 8192, 'reps': 2, 'entanglement': 'linear', 'noise_level': 'high',     'mitigation': 'ZNE+REM'},

    # =====================================================
    # SECTION B: ABLATION STUDY (Compare Techniques)
    # =====================================================
    
    # --- EXP 6: ZNE Only (for comparison) ---
    {'id': 'ZNE_Only', 'k_features': 8, 'shots': 8192, 'reps': 2, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'ZNE'},
    
    # --- EXP 7: REM Only (for comparison) ---
    {'id': 'REM_Only', 'k_features': 8, 'shots': 8192, 'reps': 2, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'REM'},
    
    # --- EXP 8: No Mitigation (baseline) ---
    {'id': 'NONE_Baseline', 'k_features': 8, 'shots': 8192, 'reps': 2, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'NONE'},
]

# Count experiments by type
zne_count = sum(1 for e in experiments if e['mitigation'] == 'ZNE')
rem_count = sum(1 for e in experiments if e['mitigation'] == 'REM')
combined_count = sum(1 for e in experiments if e['mitigation'] == 'ZNE+REM')
none_count = sum(1 for e in experiments if e['mitigation'] == 'NONE')

print(f"Total experiments configured: {len(experiments)}")
print(f"  - ZNE+REM combined: {combined_count} (PRIMARY)")
print(f"  - ZNE only: {zne_count} (ablation)")
print(f"  - REM only: {rem_count} (ablation)")
print(f"  - No mitigation: {none_count} (baseline)")
print(f"\nDefault settings: 8 features, 8192 shots, 2 reps, linear entanglement")


Total experiments configured: 19
  - ZNE+REM combined: 16 (PRIMARY)
  - ZNE only: 1 (ablation)
  - REM only: 1 (ablation)
  - No mitigation: 1 (baseline)

Default settings: 8 features, 8192 shots, 2 reps, linear entanglement


##### Main Experiment Loop (with ZNE + REM Support)

In [11]:
# ===================================================================
# MAIN EXPERIMENT LOOP - ERROR MITIGATED QSVM (ZNE + REM COMBINED)
# LUNG CANCER DATASET
# ===================================================================
import os
import time

# Setup kernel directory
kernel_dir = 'kernels_em_znerem_lungcancer'
os.makedirs(kernel_dir, exist_ok=True)

# ZNE scales for Richardson extrapolation
ZNE_SCALES = [1.0, 3.0]

all_results = []

for exp_num, config in enumerate(experiments, 1):
    print("\n" + "="*80)
    print(f"EXPERIMENT {exp_num}/{len(experiments)}: {config['id']} (ZNE+REM Mitigated)")
    print("="*80)
    
    # Check if files exist (but DON'T skip the loop!)
    train_file_scale1 = f'{kernel_dir}/kernel_train_scale1_{config["id"]}.npy'
    test_file_scale1 = f'{kernel_dir}/kernel_test_scale1_{config["id"]}.npy'
    train_file_scale3 = f'{kernel_dir}/kernel_train_scale3_{config["id"]}.npy'
    test_file_scale3 = f'{kernel_dir}/kernel_test_scale3_{config["id"]}.npy'
    
    skip_quantum = (os.path.exists(train_file_scale1) and 
                    os.path.exists(test_file_scale1) and
                    os.path.exists(train_file_scale3) and 
                    os.path.exists(test_file_scale3))
    
    # -------------------------------------------------------------------
    # DATA PREPARATION (USE ALREADY LOADED DATA FROM TOP)
    # -------------------------------------------------------------------
    X = df.drop(['label', 'label_binary'], axis=1)
    y = df['label_binary']
    
    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, random_state=42, stratify=y
    )
    
    # One-hot encoding
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    X_train_encoded = pd.DataFrame(encoder.fit_transform(X_train), columns=encoder.get_feature_names_out())
    X_test_encoded = pd.DataFrame(encoder.transform(X_test), columns=encoder.get_feature_names_out())
    
    # Cramér's V feature selection
    def cramers_v(x, y):
        confusion_matrix = pd.crosstab(x, y)
        chi2 = chi2_contingency(confusion_matrix)[0]
        n = confusion_matrix.sum().sum()
        phi2 = chi2 / n
        r, k = confusion_matrix.shape
        phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
        rcorr = r - ((r-1)**2)/(n-1)
        kcorr = k - ((k-1)**2)/(n-1)
        if min((kcorr-1), (rcorr-1)) == 0: 
            return 0
        return np.sqrt(phi2corr / min((kcorr-1), (rcorr-1)))
    
    cramers_scores = {col: cramers_v(X_train_encoded[col], y_train) for col in X_train_encoded.columns}
    cramers_series = pd.Series(cramers_scores).sort_values(ascending=False)
    
    k_features = config['k_features']
    top_features = cramers_series.head(k_features).index.tolist()
    X_train_kbest = X_train_encoded[top_features].values
    X_test_kbest = X_test_encoded[top_features].values
    selected_features = top_features
    
    # -------------------------------------------------------------------
    # KERNEL (Load OR Compute)
    # -------------------------------------------------------------------
    kernel_time = 0
    
    if skip_quantum:
        print(f"Loading existing kernels...")
        kernels_train = {
            1.0: np.load(train_file_scale1),
            3.0: np.load(train_file_scale3)
        }
        kernels_test = {
            1.0: np.load(test_file_scale1),
            3.0: np.load(test_file_scale3)
        }
        noise_level = config.get('noise_level', 'standard')
    else:
        print(f"Computing quantum kernels...")
        noise_level = config.get('noise_level', 'standard')
        
        kernels_train = {}
        kernels_test = {}
        
        feature_map = ZZFeatureMap(
            feature_dimension=k_features, 
            reps=config['reps'], 
            entanglement=config['entanglement']
        )
        
        start_kernel = time.time()
        for scale in ZNE_SCALES:
            _, backend, pm, _ = get_scaled_noise_model(scale_factor=scale, level=noise_level)
            sampler = AerSampler.from_backend(backend=backend, default_shots=config['shots'])
            fidelity = ComputeUncompute(sampler=sampler, pass_manager=pm)
            qkernel = FidelityQuantumKernel(fidelity=fidelity, feature_map=feature_map)
            
            kernels_train[scale] = qkernel.evaluate(x_vec=X_train_kbest)
            kernels_test[scale] = qkernel.evaluate(x_vec=X_test_kbest, y_vec=X_train_kbest)
        
        kernel_time = time.time() - start_kernel
        
        np.save(train_file_scale1, kernels_train[1.0])
        np.save(test_file_scale1, kernels_test[1.0])
        np.save(train_file_scale3, kernels_train[3.0])
        np.save(test_file_scale3, kernels_test[3.0])
    
    # -------------------------------------------------------------------
    # ERROR MITIGATION (ALWAYS RUN)
    # -------------------------------------------------------------------
    # Get noise config for REM
    noise_config = NOISE_CONFIGS.get(noise_level, NOISE_CONFIGS['standard'])
    p_ro = noise_config['p_ro']
    n_qubits = k_features
    
    # Step 1: ZNE (Linear Richardson extrapolation)
    kernel_train_zne = 1.5 * kernels_train[1.0] - 0.5 * kernels_train[3.0]
    kernel_test_zne = 1.5 * kernels_test[1.0] - 0.5 * kernels_test[3.0]
    
    # Step 2: REM (Readout error mitigation)
    matrix_train = apply_rem_to_kernel(kernel_train_zne, p_ro, n_qubits)
    matrix_test = apply_rem_to_kernel(kernel_test_zne, p_ro, n_qubits)
    
    # Ensure validity
    matrix_train = np.clip(matrix_train, 0, 1)
    matrix_test = np.clip(matrix_test, 0, 1)
    
    # -------------------------------------------------------------------
    # GRID SEARCH & EVALUATION (ALWAYS RUN)
    # -------------------------------------------------------------------
    param_grid = {'C': [0.001, 0.01, 0.1, 1, 10, 100]}
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    grid_search = GridSearchCV(
        SVC(kernel='precomputed', class_weight='balanced'),
        param_grid, cv=cv, scoring='accuracy', n_jobs=-1
    )
    
    start_train = time.time()
    grid_search.fit(matrix_train, y_train)
    train_time = time.time() - start_train
    
    best_model = grid_search.best_estimator_
    best_c = grid_search.best_params_['C']
    cv_score = grid_search.best_score_
    
    y_train_pred = best_model.predict(matrix_train)
    y_test_pred = best_model.predict(matrix_test)
    
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    test_balanced_acc = balanced_accuracy_score(y_test, y_test_pred)
    cancer_recall = recall_score(y_test, y_test_pred, pos_label=1)
    gen_gap = abs(train_acc - test_acc)
    
    print(f"Test Acc: {test_acc:.4f} | Recall: {cancer_recall:.4f}")
    
    # -------------------------------------------------------------------
    # STORE (ALWAYS)
    # -------------------------------------------------------------------
    all_results.append({
        'experiment_id': config['id'],
        'exp_number': exp_num,
        'k_features': k_features,
        'shots': config['shots'],
        'reps': config['reps'],
        'entanglement': config['entanglement'],
        'noise_level': noise_level,
        'selected_features': selected_features,
        'best_c': best_c,
        'cv_score': cv_score,
        'train_acc': train_acc,
        'test_acc': test_acc,
        'test_balanced_acc': test_balanced_acc,
        'cancer_recall': cancer_recall,
        'gen_gap': gen_gap,
        'kernel_time': kernel_time,
        'train_time': train_time
    })

print("\n" + "="*80)
print("COMPLETE!")
print("="*80)

results_df = pd.DataFrame(all_results)
results_df.to_csv('em_qsvm_lungcancer_znerem_results.csv', index=False)
print(f"Saved {len(results_df)} experiments to {kernel_dir}/")



EXPERIMENT 1/19: ZNEREM_4feat (ZNE+REM Mitigated)
Computing quantum kernels...


/tmp/ipykernel_17/230892927.py:95: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(


Test Acc: 0.6000 | Recall: 0.5714

EXPERIMENT 2/19: ZNEREM_6feat (ZNE+REM Mitigated)
Computing quantum kernels...


/tmp/ipykernel_17/230892927.py:95: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(


Test Acc: 0.5000 | Recall: 0.4286

EXPERIMENT 3/19: ZNEREM_8feat (ZNE+REM Mitigated)
Computing quantum kernels...


...

Test Acc: 0.8000 | Recall: 1.0000

EXPERIMENT 18/19: REM_Only (ZNE+REM Mitigated)
Computing quantum kernels...


/tmp/ipykernel_17/230892927.py:95: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(


Test Acc: 0.8000 | Recall: 1.0000

EXPERIMENT 19/19: NONE_Baseline (ZNE+REM Mitigated)
Computing quantum kernels...


/tmp/ipykernel_17/230892927.py:95: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(


Test Acc: 0.8000 | Recall: 1.0000

COMPLETE!
Saved 19 experiments to kernels_em_znerem_lungcancer/


##### Results Summary and Visualization

In [12]:
# Display summary table
print("\nResults Summary:")
print(results_df[['id', 'mitigation', 'k_features', 'noise_level', 'test_acc', 'recall', 'gen_gap']].to_string(index=False))


Results Summary:


KeyError: "['id', 'mitigation', 'recall'] not in index"

In [ ]:
# ==========================================
# VISUALIZATION: Compare Mitigation Methods
# ==========================================

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

# Filter to comparable experiments (10 features, standard noise)
baseline_filter = (results_df['k_features'] == 10) & (results_df['noise_level'] == 'standard')
baseline_df = results_df[baseline_filter].copy()

if len(baseline_df) > 0:
    # Group by mitigation type
    mitigation_summary = baseline_df.groupby('mitigation').agg({
        'test_acc': 'mean',
        'recall': 'mean',
        'gen_gap': 'mean'
    }).reset_index()
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    colors = {'NONE': '#d62728', 'ZNE': '#1f77b4', 'REM': '#ff7f0e', 'ZNE+REM': '#2ca02c'}
    bar_colors = [colors.get(m, '#333333') for m in mitigation_summary['mitigation']]
    
    axes[0].bar(mitigation_summary['mitigation'], mitigation_summary['test_acc'], color=bar_colors)
    axes[0].set_ylabel('Test Accuracy')
    axes[0].set_title('Test Accuracy by Mitigation Method')
    axes[0].set_ylim(0, 1)
    
    axes[1].bar(mitigation_summary['mitigation'], mitigation_summary['recall'], color=bar_colors)
    axes[1].set_ylabel('Recall')
    axes[1].set_title('Recall by Mitigation Method')
    axes[1].set_ylim(0, 1)
    
    axes[2].bar(mitigation_summary['mitigation'], mitigation_summary['gen_gap'], color=bar_colors)
    axes[2].set_ylabel('Generalization Gap')
    axes[2].set_title('Generalization Gap (Lower is Better)')
    
    plt.tight_layout()
    plt.savefig('lungcancer_mitigation_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\nMitigation Method Summary:")
    print(mitigation_summary.to_string(index=False))
else:
    print("No baseline experiments found for visualization.")

In [ ]:
# ==========================================
# HEATMAP: All Experiments Overview
# ==========================================

plt.figure(figsize=(12, 10))

heatmap_data = results_df.set_index('id')[['test_acc', 'recall', 'gen_gap', 'train_acc']]

sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='RdYlGn', 
            center=0.75, linewidths=.5, cbar_kws={'label': 'Score'})

plt.title('Error Mitigation Experiments - Lung Cancer (ZNE + REM)', fontsize=14, pad=20)
plt.ylabel('Experiment ID')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('lungcancer_em_experiments_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()